In [7]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("yolov9_paper.pdf")
data = loader.load()  
#data

In [9]:
len(data)

18

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

#split data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(data)


print("Total no of documents: ",len(docs))

Total no of documents:  96


In [11]:
docs[1]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-03-01T01:40:26+00:00', 'author': '', 'keywords': '', 'moddate': '2024-03-01T01:40:26+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'yolov9_paper.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='when input data undergoes layer-by-layer feature extrac-\ntion and spatial transformation, large amount of informa-\ntion will be lost. This paper will delve into the important is-\nsues of data loss when data is transmitted through deep net-\nworks, namely information bottleneck and reversible func-\ntions. We proposed the concept of programmable gradi-\nent information (PGI) to cope with the various changes\nrequired by deep networks to achieve multiple objectives.\nPGI can provide complete input information for the tar-\nget task to calculat

In [15]:
%pip install -U langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4920.86it/s]


In [17]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [18]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [20]:
retrieved_docs = retriever.invoke("What is new in YOLOv9?")

print(len(retrieved_docs))
print(retrieved_docs[0].page_content)

5
YOLO MS-N [7] 4.5 17.4 43.4 60.4 47.6 23.7 48.3 60.3
YOLO MS-S [7] 8.1 31.2 46.2 63.7 50.5 26.9 50.5 63.0
YOLO MS [7] 22.2 80.2 51.0 68.6 55.7 33.1 56.1 66.5
GELAN-S (Ours) 7.1 26.4 46.7 63.0 50.7 25.9 51.5 64.0
GELAN-M (Ours) 20.0 76.3 51.1 67.9 55.7 33.6 56.4 67.3
GELAN-C (Ours) 25.3 102.1 52.5 69.5 57.3 35.8 57.6 69.4
GELAN-E (Ours) 57.3 189.0 55.0 71.9 60.0 38.0 60.6 70.9
YOLOv9-S (Ours) 7.1 26.4 46.8 63.4 50.7 26.6 56.0 64.5
YOLOv9-M (Ours) 20.0 76.3 51.4 68.1 56.1 33.6 57.0 68.0
YOLOv9-C (Ours) 25.3 102.1 53.0 70.2 57.8 36.2 58.5 69.3
YOLOv9-E (Ours) 57.3 189.0 55.6 72.8 60.6 40.2 61.0 71.4
5.3. Comparison with state-of-the-arts
Table 1 lists comparison of our proposed YOLOv9 with
other train-from-scratch real-time object detectors. Over-
all, the best performing methods among existing methods
are YOLO MS-S [7] for lightweight models, YOLO MS [7]
for medium models, YOLOv7 AF [63] for general mod-
els, and YOLOv8-X [15] for large models. Compared with


In [24]:
%pip install -U langchain-huggingface huggingface_hub

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: uuid-utils<1.0,>=0.12.0 in c:\users\vasur\anaconda3\envs\env_langchain1\lib\site-packages (from langchain-core<2.0.0,>=1.2.31->langchain-huggingface) (0.17.0)



In [44]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter Hugging Face token:")

Enter Hugging Face token: ········


In [47]:
from huggingface_hub import whoami
import os

print(whoami(token=os.environ["HF_TOKEN"]))

{'type': 'user', 'id': '6a70b17b804bb9057eb61375', 'name': 'vasu0312', 'fullname': 'Vasudeva Reddy Bolleddula', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/0842b63ecb122d5266e974558a29dec5.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'vasudeva', 'role': 'fineGrained', 'createdAt': '2026-08-03T15:31:47.329Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write', 'post.write'], 'scoped': [{'entity': {'_id': '6a70b17b804bb9057eb61375', 'type': 'user', 'name': 'vasu0312'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.write', 'inference.serverless.write', 'inference.endpoints.infer.write', 'inference.endpoints.write', 'user.webhooks.read', 'user.webhooks.write', 'collection.read', 'collection.write', 'discussion.write', 'user.billing.read', 'job.write', 'user.notifications.read', 'user.notifications.write']}]}}}}


In [49]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(
    api_key=os.environ["HF_TOKEN"]
)

completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "What is YOLO?"
        }
    ],
    max_tokens=200
)


**YOLO** can refer to two very different things, depending on the context:

---

## 1. “You Only Live Once” (pop‑culture phrase)

- **Meaning**: A catchy mantra that encourages people to seize the moment, take risks, or enjoy life because you only get one chance at it.
- **Origin**: Popularized in the early 2010s through social media, music (e.g., Drake’s 2011 song “The Motto”), and memes.
- **Usage**: Often seen in text messages, hashtags (`#YOLO`), or as a justification for spontaneous decisions—sometimes humorously, sometimes seriously.

---

## 2


In [ ]:
print(completion.choices[0].message.content)

In [51]:
%pip install -U langchain-openai

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 29.7 MB/s  0:00:00
   ---------------------------------------- 0.0/876.6 kB ? eta -:--:--
   ---------------------------------------- 876.6/876.6 kB 41.0 MB/s  0:00:00

   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   -------------------- ------------------- 2/4 [openai]
   --------------------

In [61]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=os.environ["HF_TOKEN"],
    base_url="https://router.huggingface.co/v1",
    temperature=0.1,
    max_tokens=400
)



system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [62]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [63]:
response = rag_chain.invoke(    {"input": "What is new in YOLOv9?"})

print(response["answer"])

YOLOv9 introduces a new backbone built from CSP‑ELAN and SPP‑ELAN modules, a streamlined training schedule (500 epochs with SGD, linear LR decay from 0.01 to 0.0001, 3‑epoch warm‑up and mosaic turned off in the last 15 epochs), and a set of tuned augmentations (HSV, mosaic, MixUp, copy‑&‑paste). These changes give it higher AP and speed across lightweight, medium, and large models than prior train‑from‑scratch detectors such as YOLO MS and YOLOv7 AF. The result is a family of YOLOv9‑S/M/C/E models that consistently outperform comparable real‑time detectors.
